# evoproto quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omega2417/bnt/blob/master/evoproto/notebooks/01_quickstart.ipynb)

Reference implementation of the evolution-informed, evidence-gated design
framework. This notebook walks the whole pipeline once, in the order of the
article: knowledge graph → phylogenetic tests → analog scoring → constrained
search → evidence gate.

**Everything below is SYNTHETIC or MODELED.** No biological records ship with
the package, and no number here supports an empirical claim.


In [ ]:
# Google Colab: install the package from the repository (skip if running locally)
import importlib.util, subprocess, sys

if importlib.util.find_spec('evoproto') is None:
    IN_COLAB = 'google.colab' in sys.modules
    if IN_COLAB:
        subprocess.run(['pip', 'install', '-q',
                        'git+https://github.com/omega2417/bnt.git#subdirectory=evoproto'],
                       check=True)
    else:
        sys.path.insert(0, '../src')   # running from notebooks/ in a clone

import evoproto
print('evoproto', evoproto.__version__)


## 1. The evolutionary–engineering knowledge graph (Section 4.2)

Six node types, nine typed edges. `homologousTo` and `convergentWith` are
mutually exclusive between any two trait nodes — the schema refuses the second
one — and the number of independent origins is *derived* from the graph, not
asserted.


In [ ]:
from evoproto.kg import demo_graph, EdgeType

graph = demo_graph()
summary = graph.summary()
print({k: v for k, v in summary['nodes'].items() if v})
print({k: v for k, v in summary['edges'].items() if v})

mechanism = 'mech:sandwich_cellular_stiffening'
print('independent origins N_conv =', graph.independent_origins(mechanism))


In [ ]:
# Try to make two convergent traits homologous: the schema rejects it.
from evoproto.kg import SchemaError
from evoproto.data import Provenance, Tag

p = Provenance('demo', '1', 'MIT', tag=Tag.SYNTHETIC)
try:
    graph.add_edge(EdgeType.HOMOLOGOUS_TO, 'trait:diaphragmed_culm',
                   'trait:stereom_foam', 0.9, p)
except SchemaError as error:
    print('rejected:', error)


### The traceability chain (Eq. 4)

Every recommendation must expose at least one chain with all four provenance
records. The tag on each link is what keeps a MODELED number from being read
as a MEASURED one.


In [ ]:
chain = graph.evidence_chain('par:flexural_rigidity_per_mass')[0]
for link in chain.links:
    print(f'{link.edge_type.value:<12} {link.tail} -> {link.head}'
          f'  s={link.support:.2f}  {link.provenance.tag.value}')
print('measured fraction M =', chain.measured_fraction)


## 2. Phylogenetic context (Section 4.3)

Is the trait inherited or selected? Blomberg's *K* and Pagel's *λ* answer that;
Stayton's *C1* with a Brownian null decides whether a `convergentWith` edge may
be inserted at all.


In [ ]:
from evoproto.phylo import (blombergs_k, c1_significance, induce_convergence,
                            pagels_lambda, random_ultrametric_tree)

tree = random_ultrametric_tree(18, seed=20260918)
x, (tip_a, tip_b) = induce_convergence(tree, seed=20260918)

print('K      =', round(blombergs_k(tree, x), 3))
print('lambda =', round(pagels_lambda(tree, x)[0], 3))
result = c1_significance(tree, x, tip_a, tip_b, n_sim=200, seed=1)
print('C1     =', round(result['c1'], 3), ' p =', round(result['p_value'], 4))


## 3. Analog scoring and transfer uncertainty (Section 4.4)

`S = w_F·F + w_M·M + w_N·ln(1+N_conv) − w_U·U`. The logarithm is there because
the third independent origin of a mechanism adds less confidence than the second.


In [ ]:
from evoproto.retrieval import AnalogCandidate, rank_analogs

candidates = [
    AnalogCandidate('avian_long_bone', functional_match=0.82,
                    mechanism_evidence=chain.measured_fraction,
                    n_independent_origins=3, length_ratio=3.0),
    AnalogCandidate('deep_sea_sponge', functional_match=0.78,
                    mechanism_evidence=0.0, n_independent_origins=1,
                    length_ratio=300.0, material_biological='hierarchical_composite',
                    load_biological='static'),
]
for row in rank_analogs(candidates):
    print(f"{row['identifier']:<18} S={row['S']:+.3f}  U={row['U']:.2f}"
          f"  dominant: {row['dominant_uncertainty']}")


## 4. Constrained multi-objective search (Algorithm 1, Section 6)

The EOAT bracket of the case study. Uniform sampling of the design space is
about 1 % feasible under Table 6, so the constraints genuinely bind.


In [ ]:
import numpy as np
from evoproto.data import stable_seed
from evoproto.design import (BracketSpec, analog_prior_sampler, bounds_array,
                             evaluate_population, feasible_fraction_uniform)
from evoproto.optimize import constrained_search, normalized_hypervolume

spec = BracketSpec()
print('tip load P =', round(spec.tip_load, 1), 'N')
print('uniform feasible fraction =', feasible_fraction_uniform(spec, 20000, 0))

lower, upper = bounds_array(spec)
evaluator = lambda x: evaluate_population(x, spec)[:2]
result = constrained_search(evaluator, analog_prior_sampler(spec), lower, upper,
                            population=40, generations=25,
                            seed=stable_seed('T1', 'C', 0), arm='C')
pareto_x, pareto_f = result.pareto()
print('feasible fraction', float(np.mean(result.feasible)),
      '| Pareto size', len(pareto_x),
      '| HV', round(normalized_hypervolume(pareto_f, [spec.mass_max, spec.delta_max]), 3))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.scatter(result.f[:, 0] * 1e3, result.f[:, 1] * 1e6, s=14, alpha=0.7,
           label='final population')
ax.scatter(pareto_f[:, 0] * 1e3, pareto_f[:, 1] * 1e6, s=18, color='#b3402f',
           label='Pareto set')
ax.axvline(spec.mass_max * 1e3, ls='--', lw=0.8, color='grey')
ax.axhline(spec.delta_max * 1e6, ls='--', lw=0.8, color='grey')
ax.set_xlabel('mass (g)'); ax.set_ylabel('tip deflection (um)')
ax.legend(frameon=False); ax.set_title('SYNTHETIC / MODELED')
plt.show()


## 5. The evidence gate (Algorithm 2, Section 4.6)

Verification is necessary but not sufficient. When the evidence is too weak or
the transfer too uncertain, the system abstains — and the abstention names the
experiment that would fix it, which turns a gap in evidence into a research task.


In [ ]:
from evoproto.gate import decide
from evoproto.retrieval import transfer_uncertainty

u = transfer_uncertainty(3.0, 'hierarchical_composite', 'monolithic_alloy',
                         'cyclic', 'intermittent_inertial')
origins = graph.independent_origins(mechanism)

good = decide(True, chain.supports, u, chain.reconstructed, origins, chain=chain)
print(good.decision.value, '|', good.reason)
print('  next step:', good.report['next_step'])

weak = decide(True, [0.9, 0.1, 0.2, 0.3], u, chain=chain)
print(weak.decision.value, '|', weak.reason)
print('  weakest edge:', weak.report['weakest_edge'])
print('  next step:', weak.report['next_step'])

unverified = decide(False, chain.supports, u)
print(unverified.decision.value, '|', unverified.reason)


## What you just ran

The five stages above are the whole framework: graph → phylogeny → retrieval →
search → gate. The next notebooks reproduce the article's figures
(`02_reproduce_figures.ipynb`) and its protocol dry run
(`03_protocol_dry_run.ipynb`).
